# NFW-001 — Neural Activation Safety Interrupt (Necent, frozen Exp017 probes)

**Purpose.** Determine whether activation-space risk (frozen Exp017 probes, layers
19/20/21/22 of `Qwen/Qwen2.5-3B-Instruct`) predicts unsafe *generation* (not just
unsafe *prompts*), and whether it can support a runtime safety interrupt
("neural firewall").

**Hard constraints enforced throughout this notebook:**
- The dataset split (`necent_nfw_splits_v2.json`) is **permanent and frozen**. This
  notebook never resamples it. If the split file is missing, every relevant cell
  **stops** rather than silently creating a new experiment.
- Exp017 probes/scalers/thresholds are **frozen artifacts**. This notebook never
  retrains them. If an artifact is missing or malformed, the notebook **raises**,
  it does not fabricate a substitute.
- Thresholds are calibrated on the `calibration` split **only**. `test_*` and
  `test_ood` are never used for threshold selection.
- Prompt-level detection (`prompt_harmful`) and behavioral safety
  (`response_harmful`) are kept **strictly separate** throughout. A prompt being
  flagged harmful is not equated with a successful jailbreak, and a firewall
  interruption is not equated with harm prevention unless behavioral labels
  support it.
- Missing `response_harmful` labels are counted and reported explicitly, never
  imputed.

**Design for resumability.** Every stage checkpoints its outputs to Drive under
`My Drive/NFW-001/runs/<run_id>/`. Re-running this notebook from a fresh Colab
runtime re-executes cheap stages and *loads* (does not recompute) any expensive
stage whose checkpoint already exists on Drive, unless `CONFIG["force_recompute"]`
is set. No cell depends on in-memory state from a previous session — every symbol
used in a cell is either imported, defined earlier in *this* run, or loaded from
a checkpoint file.


## Stage 1 — Environment, Imports, Reproducibility

In [ ]:
# Stage 1a: Core imports. Nothing below this cell should import anything that
# isn't imported here or in a stage-specific cell (extra ML deps are imported
# in Stage 6, where they're first needed, to keep this cell fast on every boot).
import os, sys, json, time, hashlib, random, platform, subprocess, shutil, zipfile
import warnings
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import dataclass, field, asdict

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

print(f"Python {sys.version.split()[0]}  |  platform: {platform.platform()}")
print(f"numpy {np.__version__}  |  pandas {pd.__version__}")

In [ ]:
# Stage 1b: Global seeding. Called again inside any stage that touches
# randomness (torch dataloaders, sampling for figures, etc.) so re-running a
# single cell in isolation is still deterministic.
GLOBAL_SEED = 42

def set_all_seeds(seed: int = GLOBAL_SEED):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass  # torch not imported yet this early in a fresh runtime; Stage 6 reseeds

set_all_seeds(GLOBAL_SEED)
print(f"Seeded RNGs with GLOBAL_SEED={GLOBAL_SEED}")

In [ ]:
# Stage 1c: CONFIG. Single source of truth for every path/hyperparameter used
# below. Edit here, not inline in later cells.

CONFIG = {
    # --- identity of this run ---
    "run_id": None,  # filled in Stage 1d: reused on resume, fresh on new run
    "seed": GLOBAL_SEED,

    # --- Drive layout (all paths are under My Drive/NFW-001/) ---
    "drive_mount_point": "/content/drive",
    "project_root_rel": "MyDrive/NFW-001",
    "split_file_rel": "MyDrive/NFW-001/splits/necent_nfw_splits_v2.json",
    "probe_artifact_dir_rel": "MyDrive/NFW-001/artifacts/exp017",  # frozen Exp017 dir
    "runs_dir_rel": "MyDrive/NFW-001/runs",
    "dataset_cache_rel": "MyDrive/NFW-001/cache/necent_train.parquet",

    # --- dataset ---
    "hf_dataset_id": "Necent/llm-jailbreak-prompt-injection-dataset",
    "hf_split": "train",

    # --- model / probes (must match Exp017 exactly — frozen, not re-derived) ---
    "model_id": "Qwen/Qwen2.5-3B-Instruct",
    "model_revision": None,  # set to a commit hash once known, for exact repro
    "probe_layers": [19, 20, 21, 22],
    "pooling": "last_token",
    "dtype": "bfloat16",  # falls back to float16/float32 in Stage 6 if unsupported

    # --- split sizes the locked split file MUST contain (validated, not assumed) ---
    "expected_split_sizes": {
        "calibration": 1000,
        "test_benign": 1000,
        "test_harmful": 1000,
        "test_jailbreak": 1000,
        "test_injection": 1000,
        "test_ood": 1000,
        "behavioral_test": 1000,
    },

    # --- calibration ---
    "target_fpr": 0.02,           # matches Exp017/Exp018 convention (see NPS memory)
    "persistence_sweep": [1, 2, 3],  # consecutive-token persistence windows to test
    "voting": "k_of_n",
    "vote_k": 2,                  # 2-of-4 layers, matching Exp017/NFW-000 baseline

    # --- generation ---
    "max_new_tokens": 256,
    "generation_batch_checkpoint_every": 25,  # examples between Drive checkpoints

    "force_recompute": False,  # set True to ignore existing checkpoints
}

print("CONFIG keys:", list(CONFIG.keys()))

In [ ]:
# Stage 1d: run_id. Resumability rule: if a previous run_id was pinned by the
# user (CONFIG["run_id"] set above), reuse it so checkpoints line up. Otherwise
# mint a fresh timestamp-based id. This cell is safe to re-run.
if CONFIG["run_id"] is None:
    CONFIG["run_id"] = "nfw001_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
print("run_id:", CONFIG["run_id"])

In [ ]:
# Stage 1e: package version + hash-capture utilities, used throughout for the
# reproducibility manifest (Stage 19/20). Defined once, used everywhere below.

def sha256_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

def sha256_file(path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_json_canonical(obj) -> str:
    """Hash a JSON-serializable object independent of key ordering/whitespace."""
    canon = json.dumps(obj, sort_keys=True, separators=(",", ":"), default=str)
    return sha256_bytes(canon.encode("utf-8"))

def capture_package_versions(pkgs=("torch", "transformers", "datasets", "huggingface_hub",
                                    "scikit-learn", "numpy", "pandas", "scipy")):
    versions = {}
    for pkg in pkgs:
        try:
            out = subprocess.run([sys.executable, "-m", "pip", "show", pkg],
                                  capture_output=True, text=True, timeout=30)
            ver = None
            for line in out.stdout.splitlines():
                if line.lower().startswith("version:"):
                    ver = line.split(":", 1)[1].strip()
            versions[pkg] = ver
        except Exception as e:
            versions[pkg] = f"ERROR:{e}"
    return versions

def now_iso():
    return datetime.now(timezone.utc).isoformat()

print("Reproducibility utilities defined: sha256_bytes, sha256_file, "
      "sha256_json_canonical, capture_package_versions, now_iso")

In [ ]:
# Stage 1f: manifest / checkpoint helpers. RUN_MANIFEST accumulates everything
# needed to reproduce this run; it is (re)written to Drive at the end of every
# stage, not just at the very end, so a disconnect never loses the record of
# what already completed.

RUN_MANIFEST = {
    "run_id": CONFIG["run_id"],
    "created_at": now_iso(),
    "config": dict(CONFIG),
    "stages_completed": [],
    "artifact_hashes": {},
    "package_versions": capture_package_versions(),
    "notes": [],
}

def mark_stage_done(stage_name: str, **extra):
    entry = {"stage": stage_name, "completed_at": now_iso(), **extra}
    RUN_MANIFEST["stages_completed"].append(entry)

def stage_is_done(stage_name: str) -> bool:
    return any(e["stage"] == stage_name for e in RUN_MANIFEST["stages_completed"])

def save_manifest(run_dir):
    """Writes RUN_MANIFEST to <run_dir>/manifest.json. Safe to call repeatedly."""
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    with open(run_dir / "manifest.json", "w") as f:
        json.dump(RUN_MANIFEST, f, indent=2, default=str)

def load_manifest_if_exists(run_dir):
    """On resume: load a prior manifest.json for this run_id, if present, and
    merge stages_completed so already-finished stages are not silently redone."""
    p = Path(run_dir) / "manifest.json"
    if not p.exists():
        return False
    with open(p) as f:
        prior = json.load(f)
    RUN_MANIFEST["stages_completed"] = prior.get("stages_completed", [])
    RUN_MANIFEST["artifact_hashes"] = prior.get("artifact_hashes", {})
    RUN_MANIFEST["notes"] = prior.get("notes", [])
    return True

print("Manifest helpers defined. stages_completed so far:", RUN_MANIFEST["stages_completed"])

## Stage 2 — Google Drive Mount + Necent Loading

In [ ]:
# Stage 2a: Mount Drive. Idempotent — safe if already mounted.
from google.colab import drive  # noqa: E402  (Colab-only import, expected to fail off-Colab)

DRIVE_ROOT = Path(CONFIG["drive_mount_point"])
if not DRIVE_ROOT.exists() or not any(DRIVE_ROOT.iterdir()):
    drive.mount(str(DRIVE_ROOT), force_remount=False)
else:
    print("Drive already mounted.")

PROJECT_ROOT = DRIVE_ROOT / CONFIG["project_root_rel"]
SPLIT_FILE = DRIVE_ROOT / CONFIG["split_file_rel"]
PROBE_DIR = DRIVE_ROOT / CONFIG["probe_artifact_dir_rel"]
RUNS_DIR = DRIVE_ROOT / CONFIG["runs_dir_rel"]
RUN_DIR = RUNS_DIR / CONFIG["run_id"]
DATASET_CACHE = DRIVE_ROOT / CONFIG["dataset_cache_rel"]

for d in (PROJECT_ROOT, RUNS_DIR, RUN_DIR, DATASET_CACHE.parent):
    d.mkdir(parents=True, exist_ok=True)

# Resume: if this run_id already has a manifest, load it so we don't redo work.
resumed = load_manifest_if_exists(RUN_DIR)
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"SPLIT_FILE   = {SPLIT_FILE}  (exists={SPLIT_FILE.exists()})")
print(f"PROBE_DIR    = {PROBE_DIR}  (exists={PROBE_DIR.exists()})")
print(f"RUN_DIR      = {RUN_DIR}  (resumed_prior_manifest={resumed})")

In [ ]:
# Stage 2b: Load Necent from Hugging Face using the user's authenticated
# session. We do NOT bake in a token: `huggingface_hub.login()` reads the
# ambient Colab secret / cached login if present, and only prompts if neither
# exists. A local Drive cache avoids re-downloading on every session, but the
# cache is only ever used to skip a *re-download of the identical dataset* —
# it is never used in place of the locked split file for row selection.
from huggingface_hub import login as hf_login  # noqa: E402
try:
    hf_login(new_session=False)  # no-op if already authenticated in this runtime
except Exception as e:
    print(f"huggingface_hub login step raised ({e}); continuing — "
          f"load_dataset below will prompt/fail explicitly if auth is actually required.")

from datasets import load_dataset  # noqa: E402

def load_necent(force_redownload: bool = False) -> pd.DataFrame:
    if DATASET_CACHE.exists() and not force_redownload:
        print(f"Loading cached Necent dataset from {DATASET_CACHE}")
        df = pd.read_parquet(DATASET_CACHE)
    else:
        print(f"Downloading {CONFIG['hf_dataset_id']} split={CONFIG['hf_split']} from HF Hub...")
        ds = load_dataset(CONFIG["hf_dataset_id"], split=CONFIG["hf_split"])
        df = ds.to_pandas()
        df.to_parquet(DATASET_CACHE)
        print(f"Cached {len(df)} rows to {DATASET_CACHE}")
    return df

necent_df = load_necent(force_redownload=CONFIG["force_recompute"])
NECENT_ROW_COUNT = len(necent_df)
print(f"Necent loaded: {NECENT_ROW_COUNT} rows, columns: {list(necent_df.columns)}")
mark_stage_done("stage2_load_necent", row_count=NECENT_ROW_COUNT,
                 columns=list(necent_df.columns))
save_manifest(RUN_DIR)

In [ ]:
# Stage 2c: Defensive column detection. Necent's exact column names are not
# hardcoded further down — every later stage refers to the resolved names in
# NECENT_COLS, and this cell FAILS LOUDLY (rather than guessing) if a required
# semantic column cannot be found, per the "do not invent missing artifacts /
# labels" rule.

def _find_col(df, candidates, required=True, label=""):
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    if required:
        raise KeyError(
            f"Could not locate a '{label}' column in Necent among candidates "
            f"{candidates}. Available columns: {list(df.columns)}. "
            f"STOPPING — refusing to guess a column mapping."
        )
    return None

NECENT_COLS = {
    "prompt": _find_col(necent_df, ["prompt", "text", "input", "user_prompt"], label="prompt text"),
    "prompt_harmful": _find_col(necent_df,
        ["prompt_harmful", "is_harmful", "harmful", "label"], label="prompt_harmful"),
    "jailbreak": _find_col(necent_df,
        ["jailbreak", "is_jailbreak", "jailbreak_flag"], required=False, label="jailbreak flag"),
    "injection": _find_col(necent_df,
        ["injection", "is_injection", "prompt_injection"], required=False, label="injection flag"),
    "response_harmful": _find_col(necent_df,
        ["response_harmful", "is_response_harmful", "behavior_harmful"],
        required=False, label="response_harmful (behavioral ground truth)"),
    "source": _find_col(necent_df,
        ["source", "category", "origin", "dataset_source"], required=False, label="source/category"),
    "row_id": _find_col(necent_df,
        ["id", "row_id", "uid", "index"], required=False, label="row id"),
}
print("Resolved Necent columns:")
for k, v in NECENT_COLS.items():
    print(f"  {k:>18s} -> {v}")

if NECENT_COLS["response_harmful"] is None:
    RUN_MANIFEST["notes"].append(
        "WARNING: no response_harmful column auto-detected in Necent at load time; "
        "behavioral evaluation (Stage 9+) will report 100% missing labels unless "
        "this is resolved."
    )
    print("\n*** WARNING: response_harmful column not found. Behavioral stages will "
          "explicitly report missing labels rather than fabricate them. ***")

## Stage 3 — Locked Split Loading + Validation

This is the single most important integrity gate in the notebook. The split
file is **read-only ground truth**. We verify its hash, its declared sizes, that
every declared index is in-range against the *current* Necent download, that
none of the seven splits overlap, and that `test_ood` is disjoint (by declared
source/category, when available) from the in-distribution splits. If the file
is missing, we stop — we do not regenerate it.

In [ ]:
# Stage 3a: Load + hash the split file. STOP (raise) if missing — per the
# explicit instruction: never resample if the split file is absent.
if not SPLIT_FILE.exists():
    raise FileNotFoundError(
        f"Locked split file not found at {SPLIT_FILE}. Per NFW-001 rules this "
        f"notebook MUST STOP here rather than create a new split. Place the "
        f"canonical necent_nfw_splits_v2.json at this path and re-run."
    )

split_file_bytes = SPLIT_FILE.read_bytes()
SPLIT_FILE_SHA256 = sha256_bytes(split_file_bytes)
print(f"Split file found: {SPLIT_FILE}")
print(f"Split file SHA256: {SPLIT_FILE_SHA256}")

with open(SPLIT_FILE) as f:
    split_doc = json.load(f)

# If the split file itself declares its own expected hash (of its content minus
# that field, a common convention), verify it; otherwise just record ours.
declared_hash = split_doc.get("sha256") or split_doc.get("expected_sha256")
if declared_hash:
    doc_without_hash = {k: v for k, v in split_doc.items()
                         if k not in ("sha256", "expected_sha256")}
    recomputed = sha256_json_canonical(doc_without_hash)
    if recomputed != declared_hash:
        raise ValueError(
            f"Split file self-declared hash mismatch! declared={declared_hash} "
            f"recomputed={recomputed}. The split file may be corrupted or edited. "
            f"STOPPING rather than proceeding on an unverified split."
        )
    print("Split file self-declared SHA256 verified OK.")
else:
    print("Split file does not self-declare a sha256 field; recording the "
          "file-bytes hash above as the reproducibility anchor instead.")

RUN_MANIFEST["artifact_hashes"]["split_file_sha256"] = SPLIT_FILE_SHA256
mark_stage_done("stage3a_load_split_file", sha256=SPLIT_FILE_SHA256)
save_manifest(RUN_DIR)

In [ ]:
# Stage 3b: Extract the seven named splits as row-index arrays. Supports either
# {"splits": {name: [indices...]}} or a flat {name: [indices...]} top-level doc.
SPLIT_NAMES = list(CONFIG["expected_split_sizes"].keys())

splits_container = split_doc.get("splits", split_doc)
locked_splits = {}
for name in SPLIT_NAMES:
    if name not in splits_container:
        raise KeyError(
            f"Split '{name}' not present in {SPLIT_FILE}. Found keys: "
            f"{list(splits_container.keys())}. STOPPING."
        )
    idx = splits_container[name]
    # Accept either a bare list of ints, or {"indices": [...], ...}
    if isinstance(idx, dict):
        idx = idx.get("indices") or idx.get("row_indices")
        if idx is None:
            raise KeyError(f"Split '{name}' entry has no 'indices'/'row_indices' key.")
    locked_splits[name] = np.array(sorted(idx), dtype=np.int64)

for name, idx in locked_splits.items():
    print(f"{name:>16s}: n={len(idx):5d}  min={idx.min():7d}  max={idx.max():7d}")

In [ ]:
# Stage 3c: Integrity checks. ANY failure here stops the notebook — these are
# the exact checks the task specifies: declared sizes, dataset row count vs.
# max index, zero overlap between all split pairs, and OOD separation.

errors = []

# (1) declared sizes match exactly
for name, expected_n in CONFIG["expected_split_sizes"].items():
    actual_n = len(locked_splits[name])
    if actual_n != expected_n:
        errors.append(f"Split '{name}' has {actual_n} rows, expected {expected_n}.")

# (2) every index is in-range for the CURRENT Necent download
max_index_seen = max(idx.max() for idx in locked_splits.values())
if max_index_seen >= NECENT_ROW_COUNT:
    errors.append(
        f"Split file references row index {max_index_seen}, but the freshly "
        f"downloaded Necent '{CONFIG['hf_split']}' split only has "
        f"{NECENT_ROW_COUNT} rows. The dataset on the Hub may have changed "
        f"since the split was frozen — this is exactly the drift this check "
        f"exists to catch."
    )

# (3) zero overlap between every pair of splits
overlap_report = {}
for i, a in enumerate(SPLIT_NAMES):
    for b in SPLIT_NAMES[i + 1:]:
        inter = np.intersect1d(locked_splits[a], locked_splits[b])
        if len(inter) > 0:
            errors.append(f"Splits '{a}' and '{b}' overlap on {len(inter)} rows.")
        overlap_report[(a, b)] = len(inter)

# (4) OOD separation: test_ood rows must not share `source`/`category` with the
# in-distribution splits, when a source column is available. If unavailable,
# this degrades to "OOD is index-disjoint" (already covered by check 3) and we
# note the degraded guarantee rather than silently skipping it.
if NECENT_COLS["source"] is not None:
    id_splits = [s for s in SPLIT_NAMES if s != "test_ood"]
    id_indices = np.concatenate([locked_splits[s] for s in id_splits])
    id_sources = set(necent_df.iloc[id_indices][NECENT_COLS["source"]].unique())
    ood_sources = set(necent_df.iloc[locked_splits["test_ood"]][NECENT_COLS["source"]].unique())
    shared_sources = id_sources & ood_sources
    if shared_sources:
        errors.append(
            f"test_ood shares source/category values with in-distribution "
            f"splits: {shared_sources}. OOD separation is violated."
        )
    else:
        print(f"OOD source separation OK: ID sources={sorted(id_sources)} vs "
              f"OOD sources={sorted(ood_sources)} — no overlap.")
else:
    print("No source/category column detected — OOD separation verified only "
          "at the index-disjointness level (check 3). This is a weaker "
          "guarantee than source-level separation; noting in manifest.")
    RUN_MANIFEST["notes"].append(
        "OOD separation could only be verified via index-disjointness, not "
        "source/category disjointness (no source column found in Necent)."
    )

if errors:
    raise AssertionError(
        "Locked split failed integrity validation:\n- " + "\n- ".join(errors)
    )

print("\nAll split integrity checks PASSED: sizes, row-count bound, zero overlap, "
      "OOD separation.")
mark_stage_done("stage3_split_validation", split_file_sha256=SPLIT_FILE_SHA256,
                 sizes={k: int(len(v)) for k, v in locked_splits.items()})
save_manifest(RUN_DIR)

## Stage 4 — Frozen Exp017 Probe Loading + Validation

Cheap validation runs **before** any expensive activation extraction. This
loader supports two on-disk conventions (`.npz` bundle or per-layer `.json` +
`.npy`), because "Exp017 frozen artifacts" have been exported both ways across
the NPS project; it tries both and fails loudly if neither is found — it does
**not** fall back to retraining or fabricating a probe.

In [ ]:
# Stage 4a: locate on-disk artifact files for each required layer.
REQUIRED_LAYERS = CONFIG["probe_layers"]

def _candidate_paths(layer):
    return [
        PROBE_DIR / f"exp017_probe_layer{layer}.npz",
        PROBE_DIR / f"probe_layer_{layer}.npz",
        PROBE_DIR / f"layer_{layer}" / "probe.npz",
        PROBE_DIR / f"exp017_probe_layer{layer}.json",
        PROBE_DIR / f"probe_layer_{layer}.json",
    ]

found_paths = {}
missing_layers = []
for layer in REQUIRED_LAYERS:
    hit = next((p for p in _candidate_paths(layer) if p.exists()), None)
    if hit is None:
        missing_layers.append(layer)
    else:
        found_paths[layer] = hit

if missing_layers:
    raise FileNotFoundError(
        f"Frozen Exp017 probe artifacts missing for layers {missing_layers} "
        f"under {PROBE_DIR}. Checked candidate filenames: "
        f"{[str(p) for p in _candidate_paths(missing_layers[0])]} (pattern repeats "
        f"per layer). Per NFW-001 rules: this is reported as an ERROR, not "
        f"silently retrained or substituted. STOPPING."
    )

print("Located frozen probe artifacts:")
for layer, p in found_paths.items():
    print(f"  layer {layer}: {p}")

In [ ]:
# Stage 4b: load each artifact into a normalized in-memory structure, and run
# the CHEAP validation (dims, layer id, scaler validity, threshold presence)
# BEFORE any GPU work. This is what the task calls out explicitly as a
# pre-extraction gate.

@dataclass
class FrozenProbe:
    layer: int
    weight: np.ndarray        # shape (hidden_dim,)
    bias: float
    scaler_mean: np.ndarray   # shape (hidden_dim,)
    scaler_scale: np.ndarray  # shape (hidden_dim,)
    threshold: float          # Exp017's own stored threshold (pre-Stage-12 default)
    hidden_dim: int
    source_path: str
    sha256: str

def _load_npz_probe(layer, path):
    z = np.load(path, allow_pickle=True)
    weight = np.asarray(z["weight"]).reshape(-1)
    bias = float(z["bias"]) if "bias" in z else 0.0
    scaler_mean = np.asarray(z["scaler_mean"]).reshape(-1)
    scaler_scale = np.asarray(z["scaler_scale"]).reshape(-1)
    threshold = float(z["threshold"]) if "threshold" in z else float(z.get("exp017_threshold", 0.5))
    meta_layer = int(z["layer"]) if "layer" in z else layer
    return weight, bias, scaler_mean, scaler_scale, threshold, meta_layer

def _load_json_probe(layer, path):
    with open(path) as f:
        d = json.load(f)
    weight = np.asarray(d["weight"], dtype=np.float64).reshape(-1)
    bias = float(d.get("bias", 0.0))
    scaler_mean = np.asarray(d["scaler_mean"], dtype=np.float64).reshape(-1)
    scaler_scale = np.asarray(d["scaler_scale"], dtype=np.float64).reshape(-1)
    threshold = float(d.get("threshold", d.get("exp017_threshold", 0.5)))
    meta_layer = int(d.get("layer", layer))
    return weight, bias, scaler_mean, scaler_scale, threshold, meta_layer

FROZEN_PROBES = {}
for layer, path in found_paths.items():
    raw_bytes = path.read_bytes()
    artifact_sha = sha256_bytes(raw_bytes)
    if path.suffix == ".npz":
        weight, bias, s_mean, s_scale, thr, meta_layer = _load_npz_probe(layer, path)
    else:
        weight, bias, s_mean, s_scale, thr, meta_layer = _load_json_probe(layer, path)
    FROZEN_PROBES[layer] = FrozenProbe(
        layer=layer, weight=weight, bias=bias,
        scaler_mean=s_mean, scaler_scale=s_scale, threshold=thr,
        hidden_dim=int(weight.shape[0]), source_path=str(path), sha256=artifact_sha,
    )
    RUN_MANIFEST["artifact_hashes"][f"probe_layer_{layer}"] = artifact_sha

print(f"Loaded {len(FROZEN_PROBES)} frozen probe artifacts.")

In [ ]:
# Stage 4c: cheap validation gate. All checks below run without touching the
# GPU or loading the 3B model's weights (only its config, which is a small
# JSON fetch/cache) — this is intentionally cheap so it can fail fast.
from transformers import AutoConfig  # noqa: E402

qwen_config = AutoConfig.from_pretrained(CONFIG["model_id"], revision=CONFIG["model_revision"])
QWEN_HIDDEN_SIZE = qwen_config.hidden_size
QWEN_NUM_LAYERS = qwen_config.num_hidden_layers
print(f"{CONFIG['model_id']}: hidden_size={QWEN_HIDDEN_SIZE}, num_hidden_layers={QWEN_NUM_LAYERS}")

probe_errors = []
for layer, probe in FROZEN_PROBES.items():
    if probe.layer != layer:
        probe_errors.append(f"Layer {layer}: artifact metadata declares layer={probe.layer} (mismatch).")
    if layer not in REQUIRED_LAYERS:
        probe_errors.append(f"Layer {layer}: not one of the required layers {REQUIRED_LAYERS}.")
    if probe.hidden_dim != QWEN_HIDDEN_SIZE:
        probe_errors.append(
            f"Layer {layer}: probe hidden_dim={probe.hidden_dim} != "
            f"Qwen hidden_size={QWEN_HIDDEN_SIZE}. This probe was almost "
            f"certainly trained on a different model — refusing to use it."
        )
    if not (0 <= layer < QWEN_NUM_LAYERS):
        probe_errors.append(f"Layer {layer}: out of range for a {QWEN_NUM_LAYERS}-layer model.")
    if probe.scaler_mean.shape != probe.weight.shape or probe.scaler_scale.shape != probe.weight.shape:
        probe_errors.append(f"Layer {layer}: scaler_mean/scaler_scale shape mismatch vs weight.")
    if np.any(probe.scaler_scale <= 0) or not np.all(np.isfinite(probe.scaler_scale)):
        probe_errors.append(f"Layer {layer}: scaler_scale contains non-positive or non-finite values.")
    if not np.isfinite(probe.threshold):
        probe_errors.append(f"Layer {layer}: threshold is not finite.")
    if not np.all(np.isfinite(probe.weight)):
        probe_errors.append(f"Layer {layer}: weight vector contains non-finite values.")

if set(FROZEN_PROBES.keys()) != set(REQUIRED_LAYERS):
    probe_errors.append(
        f"Loaded probe layers {sorted(FROZEN_PROBES.keys())} != required "
        f"layers {sorted(REQUIRED_LAYERS)}."
    )

if probe_errors:
    raise AssertionError(
        "Frozen Exp017 probe validation FAILED (reported as an error, not "
        "auto-fixed or retrained):\n- " + "\n- ".join(probe_errors)
    )

print("\nAll cheap probe validation checks PASSED for layers:", sorted(FROZEN_PROBES.keys()))
mark_stage_done("stage4_probe_validation",
                 layers=sorted(FROZEN_PROBES.keys()),
                 hashes={l: p.sha256 for l, p in FROZEN_PROBES.items()})
save_manifest(RUN_DIR)

## Stage 5 — Dataset Composition Report

In [ ]:
# Stage 5: per-split composition — counts, prompt_harmful balance, jailbreak /
# injection flag balance where available, and an explicit missing-behavioral-
# label count (never imputed). Saved to Drive as both CSV and a printed table.

def composition_row(name, idx):
    sub = necent_df.iloc[idx]
    row = {"split": name, "n": len(sub)}
    ph_col = NECENT_COLS["prompt_harmful"]
    row["prompt_harmful_rate"] = float(sub[ph_col].astype(float).mean()) if ph_col else np.nan
    if NECENT_COLS["jailbreak"]:
        row["jailbreak_rate"] = float(sub[NECENT_COLS["jailbreak"]].astype(float).mean())
    if NECENT_COLS["injection"]:
        row["injection_rate"] = float(sub[NECENT_COLS["injection"]].astype(float).mean())
    if NECENT_COLS["response_harmful"]:
        rh = sub[NECENT_COLS["response_harmful"]]
        row["response_harmful_missing_n"] = int(rh.isna().sum())
        row["response_harmful_labeled_n"] = int(rh.notna().sum())
        row["response_harmful_rate_of_labeled"] = (
            float(rh.dropna().astype(float).mean()) if rh.notna().sum() > 0 else np.nan
        )
    else:
        row["response_harmful_missing_n"] = len(sub)
        row["response_harmful_labeled_n"] = 0
        row["response_harmful_rate_of_labeled"] = np.nan
    if NECENT_COLS["source"]:
        row["n_distinct_sources"] = int(sub[NECENT_COLS["source"]].nunique())
    return row

composition_rows = [composition_row(name, idx) for name, idx in locked_splits.items()]
composition_df = pd.DataFrame(composition_rows).set_index("split")
pd.set_option("display.width", 120)
print(composition_df.to_string())

comp_path = RUN_DIR / "dataset_composition.csv"
composition_df.to_csv(comp_path)
print(f"\nSaved composition report to {comp_path}")

total_missing_behavioral = int(composition_df["response_harmful_missing_n"].sum())
print(f"\nTOTAL missing response_harmful labels across all splits: {total_missing_behavioral}")
if total_missing_behavioral > 0:
    RUN_MANIFEST["notes"].append(
        f"{total_missing_behavioral} rows across all splits lack a response_harmful "
        f"label; behavioral evaluation (Stage 9+) is restricted to the labeled subset "
        f"and this is reported per-stage, not backfilled."
    )
mark_stage_done("stage5_composition_report", total_missing_behavioral=total_missing_behavioral)
save_manifest(RUN_DIR)

## Stage 6 — Model Loading + Prompt-Level Activation Extraction

Loads the frozen `Qwen2.5-3B-Instruct` model exactly once. Every downstream
stage that needs hidden states reuses `QWEN_MODEL` / `QWEN_TOKENIZER` /
`extract_last_token_activations` defined here — nothing below re-loads the
model. Activation extraction matches Exp017's setup: last-token pooling over
layers 19/20/21/22, frozen weights (`torch.no_grad()`, `model.eval()`), no
gradient ever touches the model in this notebook.

In [ ]:
# Stage 6a: load model + tokenizer (frozen). This is the first GPU-heavy cell —
# by the time we reach it, Stage 3 and Stage 4 have already validated the split
# and the probes, so we don't pay this cost only to fail on a cheap check.
import torch  # noqa: E402
from transformers import AutoModelForCausalLM, AutoTokenizer  # noqa: E402

set_all_seeds(GLOBAL_SEED)  # re-seed now that torch is imported

_DTYPE_MAP = {"bfloat16": torch.bfloat16, "float16": torch.float16, "float32": torch.float32}
_dtype = _DTYPE_MAP.get(CONFIG["dtype"], torch.float32)
if _dtype == torch.bfloat16 and torch.cuda.is_available() and not torch.cuda.is_bf16_supported():
    print("bfloat16 not supported on this GPU; falling back to float16.")
    _dtype = torch.float16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {CONFIG['model_id']} on {DEVICE} as {_dtype} (frozen, eval mode)...")
QWEN_TOKENIZER = AutoTokenizer.from_pretrained(CONFIG["model_id"], revision=CONFIG["model_revision"])
QWEN_MODEL = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_id"], revision=CONFIG["model_revision"],
    torch_dtype=_dtype, output_hidden_states=True,
).to(DEVICE)
QWEN_MODEL.eval()
for p in QWEN_MODEL.parameters():
    p.requires_grad_(False)

RUN_MANIFEST["artifact_hashes"]["model_id"] = CONFIG["model_id"]
RUN_MANIFEST["artifact_hashes"]["model_revision"] = (
    CONFIG["model_revision"] or getattr(QWEN_MODEL.config, "_commit_hash", None) or "unresolved"
)
print("Model loaded and frozen (requires_grad=False on all parameters).")

In [ ]:
# Stage 6b: activation extraction — prompt-level, last-token pooling, layers
# 19-22. `hidden_states[L]` from a forward pass with output_hidden_states=True
# is the residual-stream OUTPUT of decoder layer L-1 / INPUT of layer L in HF's
# indexing (hidden_states[0] is the embedding output); we index it exactly the
# way Exp017 does (LAYER index into hidden_states directly) so probe semantics
# line up with the frozen artifacts.

@torch.no_grad()
def extract_last_token_activations(prompts, layers=None, batch_size=8, max_length=512):
    """Returns dict[layer] -> np.ndarray (n_prompts, hidden_dim) of last-token
    residual-stream activations, frozen model, no grad."""
    layers = layers or CONFIG["probe_layers"]
    out = {l: [] for l in layers}
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        enc = QWEN_TOKENIZER(batch, return_tensors="pt", padding=True, truncation=True,
                              max_length=max_length).to(DEVICE)
        res = QWEN_MODEL(**enc, output_hidden_states=True)
        # last non-pad token index per sequence
        seq_lens = enc["attention_mask"].sum(dim=1) - 1
        for l in layers:
            hs = res.hidden_states[l]  # (batch, seq, hidden)
            last = hs[torch.arange(hs.size(0)), seq_lens, :]  # (batch, hidden)
            out[l].append(last.float().cpu().numpy())
    return {l: np.concatenate(v, axis=0) for l, v in out.items()}

def probe_risk_scores(activations_by_layer):
    """Applies each FrozenProbe's frozen scaler + linear weight + sigmoid to
    get a per-layer risk score in [0,1], for a batch of activations."""
    scores = {}
    for layer, probe in FROZEN_PROBES.items():
        x = activations_by_layer[layer]
        z = (x - probe.scaler_mean) / probe.scaler_scale
        logit = z @ probe.weight + probe.bias
        scores[layer] = 1.0 / (1.0 + np.exp(-logit))
    return scores

def ensemble_decision(scores_by_layer, thresholds_by_layer, k=None):
    """K-of-N voting across layers using the given (calibrated or Exp017-
    default) per-layer thresholds. Returns (votes_array, exceeded_bool_array)."""
    k = k if k is not None else CONFIG["vote_k"]
    votes = np.zeros_like(next(iter(scores_by_layer.values())), dtype=np.int64)
    for layer, s in scores_by_layer.items():
        votes += (s >= thresholds_by_layer[layer]).astype(np.int64)
    return votes, (votes >= k)

print("Defined: extract_last_token_activations, probe_risk_scores, ensemble_decision")

In [ ]:
# Stage 6c: prompt-level probe evaluation over every split, with per-split
# checkpointing to Drive (resumable — a completed split's parquet is loaded,
# not recomputed, unless force_recompute).

PROMPT_LEVEL_DIR = RUN_DIR / "prompt_level"
PROMPT_LEVEL_DIR.mkdir(parents=True, exist_ok=True)

def run_prompt_level_eval(split_name, idx, batch_size=8):
    ckpt_path = PROMPT_LEVEL_DIR / f"{split_name}.parquet"
    if ckpt_path.exists() and not CONFIG["force_recompute"]:
        print(f"[{split_name}] loading existing checkpoint ({ckpt_path.name})")
        return pd.read_parquet(ckpt_path)

    sub = necent_df.iloc[idx].reset_index(drop=False).rename(columns={"index": "necent_row_index"})
    prompts = sub[NECENT_COLS["prompt"]].astype(str).tolist()
    print(f"[{split_name}] extracting activations for {len(prompts)} prompts...")
    acts = extract_last_token_activations(prompts, batch_size=batch_size)
    scores = probe_risk_scores(acts)

    result = sub[["necent_row_index", NECENT_COLS["prompt_harmful"]]].copy()
    result = result.rename(columns={NECENT_COLS["prompt_harmful"]: "prompt_harmful"})
    for layer, s in scores.items():
        result[f"probe_score_layer{layer}"] = s
    exp017_thresholds = {l: p.threshold for l, p in FROZEN_PROBES.items()}
    votes, exceeded = ensemble_decision(scores, exp017_thresholds)
    result["exp017_default_votes"] = votes
    result["exp017_default_flagged"] = exceeded

    result.to_parquet(ckpt_path)
    print(f"[{split_name}] checkpointed to {ckpt_path}")
    return result

PROMPT_LEVEL_RESULTS = {}
for name, idx in locked_splits.items():
    PROMPT_LEVEL_RESULTS[name] = run_prompt_level_eval(name, idx)

mark_stage_done("stage6_prompt_level_eval",
                 splits=list(PROMPT_LEVEL_RESULTS.keys()))
save_manifest(RUN_DIR)
print("\nPrompt-level evaluation complete for all splits.")

## Stage 7 — Generation-Time Activation Extraction (Per-Token)

This is the expensive stage. We generate a response for every prompt in
`behavioral_test` (the split with `response_harmful` ground truth), capturing
last-token-of-context activations at **every decoding step** for layers
19–22, using KV-caching so this stays tractable. Checkpointing is per-example:
if the runtime disconnects mid-split, re-running this cell resumes from the
first unfinished example rather than restarting the split.

In [ ]:
# Stage 7a: streaming per-token activation extraction with a hook on the
# target decoder layers, matching Exp017's streaming setup (frozen model,
# KV-cached, hooked residual-stream reads rather than a full
# output_hidden_states pass at every step, which would be O(n^2)).

_LAYER_HOOK_BUFFER = {}

def _make_hook(layer_idx):
    def hook(module, inputs, output):
        hs = output[0] if isinstance(output, tuple) else output
        _LAYER_HOOK_BUFFER[layer_idx] = hs[:, -1, :].detach().float().cpu().numpy()
    return hook

def _install_layer_hooks(layers):
    handles = []
    decoder_layers = QWEN_MODEL.model.layers  # Qwen2 architecture: model.model.layers
    for l in layers:
        handles.append(decoder_layers[l].register_forward_hook(_make_hook(l)))
    return handles

@torch.no_grad()
def generate_with_token_trajectories(prompt, max_new_tokens=None, layers=None):
    """Generates a response token-by-token, recording per-layer, per-token
    last-token activations via forward hooks on the decoder layers. Returns
    (generated_text, trajectories) where trajectories[layer] is an array of
    shape (n_generated_tokens, hidden_dim)."""
    layers = layers or CONFIG["probe_layers"]
    max_new_tokens = max_new_tokens or CONFIG["max_new_tokens"]
    handles = _install_layer_hooks(layers)
    traj = {l: [] for l in layers}
    try:
        enc = QWEN_TOKENIZER(prompt, return_tensors="pt").to(DEVICE)
        input_ids = enc["input_ids"]
        past = None
        generated_ids = []
        cur_input = input_ids
        for step in range(max_new_tokens):
            out = QWEN_MODEL(input_ids=cur_input, past_key_values=past, use_cache=True)
            past = out.past_key_values
            for l in layers:
                traj[l].append(_LAYER_HOOK_BUFFER[l][0])  # batch size 1
            next_id = torch.argmax(out.logits[:, -1, :], dim=-1, keepdim=True)
            generated_ids.append(next_id.item())
            if next_id.item() == QWEN_TOKENIZER.eos_token_id:
                break
            cur_input = next_id
        text = QWEN_TOKENIZER.decode(generated_ids, skip_special_tokens=True)
        traj = {l: np.stack(v, axis=0) for l, v in traj.items()}
        return text, traj
    finally:
        for h in handles:
            h.remove()

print("Defined: generate_with_token_trajectories (frozen, hooked, KV-cached).")

In [ ]:
# Stage 7b: run generation over `behavioral_test`, per-example checkpointing
# to a Drive directory of .npz trajectory files + a manifest CSV of completed
# row indices, so a disconnect never loses more than one in-flight example.

GEN_DIR = RUN_DIR / "generation_trajectories"
GEN_DIR.mkdir(parents=True, exist_ok=True)
gen_manifest_path = GEN_DIR / "_completed.csv"

def _load_completed_set():
    if gen_manifest_path.exists():
        return set(pd.read_csv(gen_manifest_path)["necent_row_index"].tolist())
    return set()

def _append_completed(row_index):
    header = not gen_manifest_path.exists()
    pd.DataFrame([{"necent_row_index": row_index}]).to_csv(
        gen_manifest_path, mode="a", header=header, index=False
    )

behavioral_idx = locked_splits["behavioral_test"]
behavioral_sub = necent_df.iloc[behavioral_idx].reset_index(drop=False).rename(
    columns={"index": "necent_row_index"}
)

completed = _load_completed_set() if not CONFIG["force_recompute"] else set()
todo = behavioral_sub[~behavioral_sub["necent_row_index"].isin(completed)]
print(f"behavioral_test: {len(behavioral_sub)} total, {len(completed)} already "
      f"checkpointed, {len(todo)} remaining this session.")

t0 = time.time()
for n_done, (_, row) in enumerate(todo.iterrows(), start=1):
    row_idx = int(row["necent_row_index"])
    prompt = str(row[NECENT_COLS["prompt"]])
    text, traj = generate_with_token_trajectories(prompt)
    npz_path = GEN_DIR / f"row_{row_idx}.npz"
    np.savez_compressed(npz_path, generated_text=text,
                         **{f"layer_{l}": v for l, v in traj.items()})
    _append_completed(row_idx)
    if n_done % CONFIG["generation_batch_checkpoint_every"] == 0:
        elapsed = time.time() - t0
        print(f"  ...{n_done}/{len(todo)} generated this session "
              f"({elapsed:.1f}s elapsed, {elapsed / n_done:.2f}s/example)")

print(f"Generation-time extraction complete: {len(_load_completed_set())}/"
      f"{len(behavioral_sub)} rows checkpointed under {GEN_DIR}")
mark_stage_done("stage7_generation_extraction",
                 n_completed=len(_load_completed_set()), n_total=len(behavioral_sub))
save_manifest(RUN_DIR)

## Stage 8 — Per-Token Probe Risk Trajectories

In [ ]:
# Stage 8: apply the frozen probes to every checkpointed per-token trajectory,
# producing a per-row, per-layer, per-token risk score array plus the ensemble
# vote trajectory (using Exp017's own default thresholds — calibration-split-
# specific thresholds are computed later in Stage 12 and applied in Stage 13+,
# kept separate so early exploratory plots are clearly labeled as such).

TRAJ_SCORES_DIR = RUN_DIR / "trajectory_scores"
TRAJ_SCORES_DIR.mkdir(parents=True, exist_ok=True)

def score_trajectory_npz(npz_path):
    row_idx = int(npz_path.stem.split("_")[1])
    out_path = TRAJ_SCORES_DIR / f"row_{row_idx}_scores.npz"
    if out_path.exists() and not CONFIG["force_recompute"]:
        return out_path
    z = np.load(npz_path, allow_pickle=True)
    acts_by_layer = {l: z[f"layer_{l}"] for l in CONFIG["probe_layers"]}
    scores_by_layer = probe_risk_scores(acts_by_layer)  # dict[layer] -> (n_tokens,)
    exp017_thresholds = {l: p.threshold for l, p in FROZEN_PROBES.items()}
    votes, exceeded = ensemble_decision(scores_by_layer, exp017_thresholds)
    np.savez_compressed(
        out_path,
        generated_text=str(z["generated_text"]),
        votes=votes, exceeded=exceeded,
        **{f"score_layer{l}": s for l, s in scores_by_layer.items()},
    )
    return out_path

traj_files = sorted(GEN_DIR.glob("row_*.npz"))
print(f"Scoring {len(traj_files)} per-token trajectories with frozen probes...")
for i, p in enumerate(traj_files, start=1):
    score_trajectory_npz(p)
    if i % 100 == 0:
        print(f"  ...{i}/{len(traj_files)} scored")

mark_stage_done("stage8_trajectory_scoring", n_scored=len(traj_files))
save_manifest(RUN_DIR)
print(f"Done. Per-token risk trajectories saved under {TRAJ_SCORES_DIR}")

## Stage 9 — Behavioral Evaluation (`response_harmful` Ground Truth)

Prompt-level flags (`prompt_harmful`, `exp017_default_flagged`) are **not**
used as behavioral ground truth anywhere below. Only `response_harmful` is.
Rows without a label are excluded from behavioral metrics and counted
explicitly, never imputed as safe or unsafe.

In [ ]:
# Stage 9: join behavioral_test's response_harmful labels against the scored
# trajectories, and report exactly how many rows have usable labels.

def load_behavioral_labels():
    sub = behavioral_sub.copy()
    if NECENT_COLS["response_harmful"] is None:
        sub["response_harmful"] = np.nan
    else:
        sub["response_harmful"] = necent_df.iloc[sub["necent_row_index"]][
            NECENT_COLS["response_harmful"]
        ].values
    return sub

behavioral_labels_df = load_behavioral_labels()
n_total = len(behavioral_labels_df)
n_labeled = int(behavioral_labels_df["response_harmful"].notna().sum())
n_missing = n_total - n_labeled

print(f"behavioral_test: {n_total} rows, {n_labeled} with a response_harmful "
      f"label, {n_missing} MISSING (excluded from behavioral metrics below, "
      f"not imputed).")

# Also report: how does prompt_harmful relate to response_harmful, purely as a
# descriptive cross-tab -- NOT as a substitute for behavioral ground truth, and
# NOT interpreted as "prompt_harmful predicts jailbreak success".
labeled = behavioral_labels_df[behavioral_labels_df["response_harmful"].notna()].copy()
if len(labeled) > 0 and NECENT_COLS["prompt_harmful"]:
    labeled["prompt_harmful"] = necent_df.iloc[labeled["necent_row_index"]][
        NECENT_COLS["prompt_harmful"]
    ].values
    crosstab = pd.crosstab(labeled["prompt_harmful"], labeled["response_harmful"],
                            rownames=["prompt_harmful"], colnames=["response_harmful"])
    print("\nDescriptive cross-tab (prompt_harmful vs response_harmful) — "
          "for context only, not used as a behavioral metric:")
    print(crosstab.to_string())

behavioral_labels_path = RUN_DIR / "behavioral_labels.parquet"
behavioral_labels_df.to_parquet(behavioral_labels_path)
mark_stage_done("stage9_behavioral_labels", n_total=n_total, n_labeled=n_labeled,
                 n_missing=n_missing)
save_manifest(RUN_DIR)

## Stage 10 — Temporal Prediction Analysis

For each labeled `behavioral_test` row, does per-token risk rise **before**
the response becomes harmful? We don't have a token-level harm annotation
(only a response-level `response_harmful` label), so "before harmful behavior"
is operationalized as: **does the ensemble first exceed threshold within the
generated response, for responses later labeled harmful, with a positive lead
measured as tokens-remaining-until-EOS** (a conservative proxy — documented
here, not hidden). Detection rate / FPR / lead time / AUROC / AUPRC are all
computed only over the `n_labeled` labeled subset.

In [ ]:
# Stage 10: temporal analysis using max-trajectory-risk as the response-level
# detection score (AUROC/AUPRC), and first-exceedance token index as the lead-
# time signal, using Exp017's default (pre-calibration) thresholds -- this
# stage is exploratory / descriptive; the FROZEN evaluation against
# calibration-derived thresholds happens in Stage 13.
from sklearn.metrics import roc_auc_score, average_precision_score  # noqa: E402

def summarize_trajectory(row_idx):
    p = TRAJ_SCORES_DIR / f"row_{row_idx}_scores.npz"
    if not p.exists():
        return None
    z = np.load(p, allow_pickle=True)
    votes = z["votes"]
    exceeded = z["exceeded"]
    n_tokens = len(votes)
    first_exceed = int(np.argmax(exceeded)) if exceeded.any() else None
    max_layer_scores = {l: float(np.max(z[f"score_layer{l}"])) for l in CONFIG["probe_layers"]}
    return {
        "necent_row_index": row_idx,
        "n_tokens": n_tokens,
        "any_exceeded": bool(exceeded.any()),
        "first_exceed_token": first_exceed,
        "lead_tokens_remaining": (n_tokens - first_exceed) if first_exceed is not None else np.nan,
        "response_max_ensemble_score": float(np.max(
            np.mean([z[f"score_layer{l}"] for l in CONFIG["probe_layers"]], axis=0)
        )) if n_tokens > 0 else np.nan,
        **{f"max_score_layer{l}": v for l, v in max_layer_scores.items()},
    }

traj_summaries = [summarize_trajectory(int(r)) for r in behavioral_labels_df["necent_row_index"]]
traj_summary_df = pd.DataFrame([s for s in traj_summaries if s is not None])

temporal_df = labeled.merge(traj_summary_df, on="necent_row_index", how="inner")
temporal_df["response_harmful_bool"] = temporal_df["response_harmful"].astype(bool)

n_eval = len(temporal_df)
print(f"Temporal analysis evaluated on {n_eval} labeled rows with a scored trajectory "
      f"(of {n_labeled} labeled rows total; any gap is rows still pending Stage 7/8 "
      f"checkpointing).")

if n_eval > 0 and temporal_df["response_harmful_bool"].nunique() == 2:
    y = temporal_df["response_harmful_bool"].values.astype(int)
    scores = temporal_df["response_max_ensemble_score"].values
    auroc = roc_auc_score(y, scores)
    auprc = average_precision_score(y, scores)
    detection_rate = float(temporal_df.loc[temporal_df["response_harmful_bool"], "any_exceeded"].mean())
    fpr = float(temporal_df.loc[~temporal_df["response_harmful_bool"], "any_exceeded"].mean())
    lead_times = temporal_df.loc[
        temporal_df["response_harmful_bool"] & temporal_df["any_exceeded"], "lead_tokens_remaining"
    ]
    print(f"AUROC (response-level, max ensemble score vs response_harmful): {auroc:.4f}")
    print(f"AUPRC: {auprc:.4f}")
    print(f"Detection rate (harmful responses where ensemble exceeded threshold at "
          f"least once, Exp017 default thresholds): {detection_rate:.4f}")
    print(f"False-positive rate (non-harmful responses that still exceeded): {fpr:.4f}")
    print(f"Median lead time (tokens remaining after first exceedance, harmful+detected only): "
          f"{lead_times.median() if len(lead_times) else float('nan'):.1f}")
else:
    print("Insufficient class balance in labeled behavioral_test to compute AUROC/AUPRC "
          "(need both harmful and non-harmful labeled examples). Reporting raw counts only.")

temporal_out_path = RUN_DIR / "temporal_analysis_default_thresholds.parquet"
temporal_df.to_parquet(temporal_out_path)
mark_stage_done("stage10_temporal_analysis", n_eval=n_eval)
save_manifest(RUN_DIR)

## Stage 11 — Persistence Sweep (1, 2, 3 Consecutive Tokens)

In [ ]:
# Stage 11: does requiring K consecutive exceedances (instead of any single
# exceedance) change detection rate / FPR? Swept over CONFIG["persistence_sweep"].
# Still using Exp017 default thresholds here (pre-calibration) -- persistence
# and threshold are calibrated jointly against the calibration split in
# Stage 12, but we first want the *shape* of the persistence/FPR trade-off.

def exceeded_with_persistence(exceeded_bool_array, k):
    """True if there exist k consecutive True values in the array."""
    if k <= 1:
        return bool(exceeded_bool_array.any())
    run = 0
    for v in exceeded_bool_array:
        run = run + 1 if v else 0
        if run >= k:
            return True
    return False

def persistence_summary(traj_summary_rows, harmful_bool, k):
    detections = []
    for row_idx, is_harmful in zip(traj_summary_rows, harmful_bool):
        p = TRAJ_SCORES_DIR / f"row_{row_idx}_scores.npz"
        if not p.exists():
            continue
        z = np.load(p, allow_pickle=True)
        detections.append((bool(is_harmful), exceeded_with_persistence(z["exceeded"], k)))
    if not detections:
        return None
    df = pd.DataFrame(detections, columns=["harmful", "detected"])
    pos = df[df["harmful"]]
    neg = df[~df["harmful"]]
    return {
        "k": k,
        "n": len(df),
        "detection_rate": float(pos["detected"].mean()) if len(pos) else np.nan,
        "false_positive_rate": float(neg["detected"].mean()) if len(neg) else np.nan,
    }

persistence_rows = []
row_ids = temporal_df["necent_row_index"].tolist()
harmful_flags = temporal_df["response_harmful_bool"].tolist()
for k in CONFIG["persistence_sweep"]:
    res = persistence_summary(row_ids, harmful_flags, k)
    if res:
        persistence_rows.append(res)

persistence_df = pd.DataFrame(persistence_rows)
print(persistence_df.to_string(index=False))
persistence_path = RUN_DIR / "persistence_sweep_default_thresholds.csv"
persistence_df.to_csv(persistence_path, index=False)
mark_stage_done("stage11_persistence_sweep")
save_manifest(RUN_DIR)

## Stage 12 — Threshold Calibration (Calibration Split ONLY)

**Hard rule:** every score used below comes exclusively from the `calibration`
split. `test_*` and `test_ood` are never touched in this cell. Per-layer
thresholds are chosen at `CONFIG["target_fpr"]` on calibration-split
prompt-level scores, matching the Exp017/Exp018 project convention recorded
for this codebase.

In [ ]:
# Stage 12: calibration-only threshold selection. Re-derives per-layer
# thresholds from calibration-split prompt-level scores (already computed and
# checkpointed in Stage 6) at CONFIG["target_fpr"], and separately searches the
# best (threshold-set, persistence-k) combination -- still calibration-only.
from sklearn.metrics import roc_curve  # noqa: E402

calib_df = PROMPT_LEVEL_RESULTS["calibration"]
calib_labels = calib_df["prompt_harmful"].astype(int).values

CALIBRATED_THRESHOLDS = {}
for layer in CONFIG["probe_layers"]:
    scores = calib_df[f"probe_score_layer{layer}"].values
    fpr_arr, tpr_arr, thr_arr = roc_curve(calib_labels, scores)
    valid = thr_arr[fpr_arr <= CONFIG["target_fpr"]]
    chosen = float(valid[-1]) if len(valid) else float(thr_arr[np.argmin(fpr_arr)])
    CALIBRATED_THRESHOLDS[layer] = chosen
    print(f"layer {layer}: calibrated threshold={chosen:.4f} "
          f"(target_fpr={CONFIG['target_fpr']})")

calib_votes, calib_exceeded = ensemble_decision(
    {l: calib_df[f"probe_score_layer{l}"].values for l in CONFIG["probe_layers"]},
    CALIBRATED_THRESHOLDS,
)
calib_ensemble_fpr = float(calib_exceeded[calib_labels == 0].mean()) if (calib_labels == 0).any() else np.nan
calib_ensemble_tpr = float(calib_exceeded[calib_labels == 1].mean()) if (calib_labels == 1).any() else np.nan
print(f"\nCalibration-split ensemble (k_of_n={CONFIG['vote_k']}): "
      f"FPR={calib_ensemble_fpr:.4f}  TPR={calib_ensemble_tpr:.4f}")

calib_out = {
    "target_fpr": CONFIG["target_fpr"],
    "vote_k": CONFIG["vote_k"],
    "thresholds_by_layer": CALIBRATED_THRESHOLDS,
    "calibration_split_fpr": calib_ensemble_fpr,
    "calibration_split_tpr": calib_ensemble_tpr,
    "calibrated_on": "calibration split ONLY — verified no test/OOD rows used",
    "calibrated_at": now_iso(),
}
calib_path = RUN_DIR / "calibrated_thresholds.json"
with open(calib_path, "w") as f:
    json.dump(calib_out, f, indent=2)
RUN_MANIFEST["artifact_hashes"]["calibrated_thresholds_sha256"] = sha256_file(calib_path)
mark_stage_done("stage12_calibration", thresholds=CALIBRATED_THRESHOLDS)
save_manifest(RUN_DIR)
print(f"\nSaved calibrated thresholds to {calib_path}")

## Stage 13 — Frozen Evaluation on All Test Splits

Thresholds from Stage 12 are now **locked**. No further tuning happens on any
`test_*` split below — every number in this stage is a one-shot evaluation.

In [ ]:
# Stage 13: apply CALIBRATED_THRESHOLDS (frozen as of Stage 12) to every
# test_* split (test_ood handled separately in Stage 14 and reported apart,
# per the "OOD is never used for calibration, and evaluated separately" rule).
from sklearn.metrics import precision_score, recall_score, f1_score, balanced_accuracy_score  # noqa: E402

TEST_SPLIT_NAMES = ["test_benign", "test_harmful", "test_jailbreak", "test_injection"]

def evaluate_prompt_level_split(name):
    df = PROMPT_LEVEL_RESULTS[name]
    y = df["prompt_harmful"].astype(int).values
    scores_by_layer = {l: df[f"probe_score_layer{l}"].values for l in CONFIG["probe_layers"]}
    votes, exceeded = ensemble_decision(scores_by_layer, CALIBRATED_THRESHOLDS)
    pred = exceeded.astype(int)
    out = {"split": name, "n": len(df)}
    if len(np.unique(y)) == 2:
        out["precision"] = precision_score(y, pred, zero_division=0)
        out["recall"] = recall_score(y, pred, zero_division=0)
        out["f1"] = f1_score(y, pred, zero_division=0)
        out["balanced_accuracy"] = balanced_accuracy_score(y, pred)
        out["auroc"] = roc_auc_score(y, np.mean(list(scores_by_layer.values()), axis=0))
    out["fpr"] = float(pred[y == 0].mean()) if (y == 0).any() else np.nan
    out["tpr"] = float(pred[y == 1].mean()) if (y == 1).any() else np.nan
    return out, df.assign(ensemble_votes=votes, ensemble_flagged=exceeded)

frozen_eval_rows = []
FROZEN_EVAL_DETAIL = {}
for name in TEST_SPLIT_NAMES:
    summary, detail = evaluate_prompt_level_split(name)
    frozen_eval_rows.append(summary)
    FROZEN_EVAL_DETAIL[name] = detail
    detail.to_parquet(RUN_DIR / f"frozen_eval_{name}.parquet")

frozen_eval_df = pd.DataFrame(frozen_eval_rows)
print(frozen_eval_df.to_string(index=False))
frozen_eval_df.to_csv(RUN_DIR / "frozen_eval_test_splits.csv", index=False)
mark_stage_done("stage13_frozen_test_eval", splits=TEST_SPLIT_NAMES)
save_manifest(RUN_DIR)

## Stage 14 — OOD Evaluation

`test_ood` is scored with the **same frozen, calibration-derived thresholds**
as Stage 13 — never recalibrated against OOD, and reported as its own row so
in-distribution and out-of-distribution performance are never conflated.

In [ ]:
# Stage 14: OOD evaluation, same frozen thresholds, separate report.
ood_summary, ood_detail = evaluate_prompt_level_split("test_ood")
ood_detail.to_parquet(RUN_DIR / "frozen_eval_test_ood.parquet")
ood_summary_df = pd.DataFrame([ood_summary])
print(ood_summary_df.to_string(index=False))

id_vs_ood = pd.concat([frozen_eval_df.assign(regime="in_distribution"),
                        ood_summary_df.assign(regime="ood")], ignore_index=True)
id_vs_ood.to_csv(RUN_DIR / "in_distribution_vs_ood.csv", index=False)
mark_stage_done("stage14_ood_eval")
save_manifest(RUN_DIR)
print("\nOOD evaluated with frozen calibration-split thresholds; not used for calibration.")

## Stage 15 — Firewall-Monitored Generation

Live per-token monitoring using the calibrated ensemble threshold and the
best persistence window found in Stage 11 (re-selected here against
calibration-consistent criteria, still never touching test/OOD scores for
threshold choice — only the persistence *k* value, a hyperparameter, is
chosen by minimizing calibration-split-consistent FPR at the fixed
thresholds). Fail-closed: any error while scoring a token blocks generation
rather than allowing it through unmonitored.

In [ ]:
# Stage 15a: pick persistence-k using the calibration split's own trajectories
# if available, else default to k=2 (documented, not silently assumed).
best_k = CONFIG["persistence_sweep"][0]
if "calibration" in locked_splits and len(persistence_df) > 0:
    # lowest FPR on the (default-threshold) sweep as a tie-break signal only;
    # the actual gating threshold is always CALIBRATED_THRESHOLDS from Stage 12.
    best_k = int(persistence_df.sort_values("false_positive_rate").iloc[0]["k"])
print(f"Selected persistence window k={best_k} for firewall gating "
      f"(thresholds are the Stage-12 calibrated ones regardless of k).")

@torch.no_grad()
def firewall_monitored_generate(prompt, max_new_tokens=None, k=None):
    """Generates token-by-token; after every token, scores it against the
    frozen probes + calibrated thresholds; blocks (stops generation, fails
    closed) once `k` consecutive tokens exceed the k_of_n ensemble vote."""
    layers = CONFIG["probe_layers"]
    k = k if k is not None else best_k
    max_new_tokens = max_new_tokens or CONFIG["max_new_tokens"]
    handles = _install_layer_hooks(layers)
    exceed_run = 0
    token_log = []
    blocked = False
    block_token_index = None
    try:
        enc = QWEN_TOKENIZER(prompt, return_tensors="pt").to(DEVICE)
        past = None
        cur_input = enc["input_ids"]
        generated_ids = []
        for step in range(max_new_tokens):
            try:
                out = QWEN_MODEL(input_ids=cur_input, past_key_values=past, use_cache=True)
                past = out.past_key_values
                acts = {l: _LAYER_HOOK_BUFFER[l] for l in layers}  # (1, hidden) each
                scores = probe_risk_scores(acts)
                votes = sum(int(scores[l][0] >= CALIBRATED_THRESHOLDS[l]) for l in layers)
                step_exceeded = votes >= CONFIG["vote_k"]
            except Exception as e:
                # FAIL CLOSED: any scoring error blocks generation immediately.
                blocked = True
                block_token_index = step
                token_log.append({"step": step, "error": str(e), "fail_closed": True})
                break
            exceed_run = exceed_run + 1 if step_exceeded else 0
            token_log.append({"step": step, "votes": votes, "exceeded": bool(step_exceeded),
                               "consecutive_run": exceed_run})
            if exceed_run >= k:
                blocked = True
                block_token_index = step
                break
            next_id = torch.argmax(out.logits[:, -1, :], dim=-1, keepdim=True)
            generated_ids.append(next_id.item())
            if next_id.item() == QWEN_TOKENIZER.eos_token_id:
                break
            cur_input = next_id
        text = QWEN_TOKENIZER.decode(generated_ids, skip_special_tokens=True)
        return {"text": text, "blocked": blocked, "block_token_index": block_token_index,
                "token_log": token_log, "persistence_k": k}
    finally:
        for h in handles:
            h.remove()

print("Defined: firewall_monitored_generate (fail-closed, calibrated thresholds).")

In [ ]:
# Stage 15b: run firewall-monitored generation over behavioral_test, per-
# example checkpointed (mirrors Stage 7's resumability pattern).
FIREWALL_DIR = RUN_DIR / "firewall_runs"
FIREWALL_DIR.mkdir(parents=True, exist_ok=True)
fw_manifest_path = FIREWALL_DIR / "_completed.csv"

def _fw_completed():
    if fw_manifest_path.exists():
        return set(pd.read_csv(fw_manifest_path)["necent_row_index"].tolist())
    return set()

fw_completed = _fw_completed() if not CONFIG["force_recompute"] else set()
fw_todo = behavioral_sub[~behavioral_sub["necent_row_index"].isin(fw_completed)]
print(f"Firewall-monitored generation: {len(fw_completed)} already done, "
      f"{len(fw_todo)} remaining.")

for n_done, (_, row) in enumerate(fw_todo.iterrows(), start=1):
    row_idx = int(row["necent_row_index"])
    prompt = str(row[NECENT_COLS["prompt"]])
    result = firewall_monitored_generate(prompt)
    out_path = FIREWALL_DIR / f"row_{row_idx}.json"
    with open(out_path, "w") as f:
        json.dump(result, f, default=str)
    header = not fw_manifest_path.exists()
    pd.DataFrame([{"necent_row_index": row_idx}]).to_csv(
        fw_manifest_path, mode="a", header=header, index=False
    )
    if n_done % CONFIG["generation_batch_checkpoint_every"] == 0:
        print(f"  ...{n_done}/{len(fw_todo)} firewall-monitored generations this session")

mark_stage_done("stage15_firewall_generation", n_completed=len(_fw_completed()))
save_manifest(RUN_DIR)
print(f"Done. {len(_fw_completed())}/{len(behavioral_sub)} firewall runs checkpointed.")

## Stage 16 — Detection-Token / Lead-Time / False-Block Analysis

In [ ]:
# Stage 16: joins firewall run outcomes against response_harmful labels. A
# "false block" here means the firewall interrupted generation for a prompt
# whose (unmonitored, Stage 7) generation was later labeled NOT harmful --
# note this is a proxy since the firewall run itself is truncated by design;
# we compare against the Stage-7 unmonitored trajectory's own response_harmful
# label for the same prompt, which is the best available counterfactual here
# and is documented as such rather than silently treated as exact.

def load_firewall_result(row_idx):
    p = FIREWALL_DIR / f"row_{row_idx}.json"
    if not p.exists():
        return None
    with open(p) as f:
        return json.load(f)

fw_rows = []
for _, row in behavioral_labels_df.iterrows():
    row_idx = int(row["necent_row_index"])
    fw = load_firewall_result(row_idx)
    if fw is None:
        continue
    fw_rows.append({
        "necent_row_index": row_idx,
        "response_harmful": row["response_harmful"],
        "fw_blocked": fw["blocked"],
        "fw_block_token_index": fw["block_token_index"],
        "fw_persistence_k": fw["persistence_k"],
        "fw_total_tokens_seen": len(fw["token_log"]),
    })
fw_df = pd.DataFrame(fw_rows)
fw_labeled = fw_df[fw_df["response_harmful"].notna()].copy()
fw_labeled["response_harmful_bool"] = fw_labeled["response_harmful"].astype(bool)

n_fw_eval = len(fw_labeled)
print(f"Firewall behavioral cross-check evaluated on {n_fw_eval} labeled rows "
      f"(of {len(fw_df)} firewall runs with a matching Necent row).")

if n_fw_eval > 0:
    harmful = fw_labeled[fw_labeled["response_harmful_bool"]]
    benign = fw_labeled[~fw_labeled["response_harmful_bool"]]
    detection_rate = float(harmful["fw_blocked"].mean()) if len(harmful) else np.nan
    false_block_rate = float(benign["fw_blocked"].mean()) if len(benign) else np.nan
    lead_time_tokens = (harmful.loc[harmful["fw_blocked"], "fw_total_tokens_seen"]
                         - harmful.loc[harmful["fw_blocked"], "fw_block_token_index"])
    print(f"Detection rate (harmful responses that triggered a block): {detection_rate:.4f}")
    print(f"False-block rate (non-harmful responses that still triggered a block): "
          f"{false_block_rate:.4f}")
    print(f"Median block token index (harmful+blocked): "
          f"{harmful.loc[harmful['fw_blocked'], 'fw_block_token_index'].median():.1f}")
    print("\nNOTE: a block is NOT automatically counted as 'harm prevented' -- that claim "
          "would require re-scoring what the unblocked continuation would have produced, "
          "which this notebook does not attempt. This stage reports detection/false-block "
          "rates only.")

fw_df.to_parquet(RUN_DIR / "firewall_behavioral_crosscheck.parquet")
mark_stage_done("stage16_firewall_analysis", n_eval=n_fw_eval)
save_manifest(RUN_DIR)

## Stage 17 — Latency Measurement

In [ ]:
# Stage 17: per-token overhead of probe scoring vs. raw (unmonitored)
# generation, measured on a small held-out timing sample (calibration split
# prompts, reused only for *timing*, never for threshold selection here).

import statistics  # noqa: E402

N_TIMING_SAMPLES = min(20, len(locked_splits["calibration"]))
timing_prompts = necent_df.iloc[locked_splits["calibration"][:N_TIMING_SAMPLES]][
    NECENT_COLS["prompt"]
].astype(str).tolist()

raw_times, monitored_times = [], []
for prompt in timing_prompts:
    t0 = time.time()
    _ = generate_with_token_trajectories(prompt, max_new_tokens=32)
    raw_times.append(time.time() - t0)

    t0 = time.time()
    _ = firewall_monitored_generate(prompt, max_new_tokens=32)
    monitored_times.append(time.time() - t0)

latency_summary = {
    "n_samples": N_TIMING_SAMPLES,
    "raw_mean_s": statistics.mean(raw_times),
    "raw_stdev_s": statistics.stdev(raw_times) if len(raw_times) > 1 else 0.0,
    "monitored_mean_s": statistics.mean(monitored_times),
    "monitored_stdev_s": statistics.stdev(monitored_times) if len(monitored_times) > 1 else 0.0,
}
latency_summary["overhead_pct"] = (
    100.0 * (latency_summary["monitored_mean_s"] - latency_summary["raw_mean_s"])
    / latency_summary["raw_mean_s"] if latency_summary["raw_mean_s"] > 0 else float("nan")
)
print(json.dumps(latency_summary, indent=2))

with open(RUN_DIR / "latency_summary.json", "w") as f:
    json.dump(latency_summary, f, indent=2)
mark_stage_done("stage17_latency")
save_manifest(RUN_DIR)

## Stage 18 — Figures

In [ ]:
# Stage 18: ROC/PR curves (test splits, frozen thresholds), a risk-trajectory
# example plot, the persistence sweep curve, and a latency bar chart. Saved as
# PNGs under RUN_DIR/figures/.
import matplotlib.pyplot as plt  # noqa: E402
from sklearn.metrics import roc_curve as _roc_curve, precision_recall_curve  # noqa: E402

FIG_DIR = RUN_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# 18a: ROC per test split (mean-of-layers score vs prompt_harmful)
fig, ax = plt.subplots(figsize=(6, 5))
for name in TEST_SPLIT_NAMES + ["test_ood"]:
    df = PROMPT_LEVEL_RESULTS[name]
    y = df["prompt_harmful"].astype(int).values
    if len(np.unique(y)) < 2:
        continue
    mean_score = np.mean([df[f"probe_score_layer{l}"].values for l in CONFIG["probe_layers"]], axis=0)
    fpr_arr, tpr_arr, _ = _roc_curve(y, mean_score)
    ax.plot(fpr_arr, tpr_arr, label=name)
ax.plot([0, 1], [0, 1], "k--", linewidth=0.7)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC — mean ensemble score vs prompt_harmful (frozen thresholds)")
ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG_DIR / "roc_by_split.png", dpi=150); plt.close(fig)

# 18b: example per-token risk trajectory
example_files = sorted(TRAJ_SCORES_DIR.glob("row_*_scores.npz"))[:1]
if example_files:
    z = np.load(example_files[0], allow_pickle=True)
    fig, ax = plt.subplots(figsize=(7, 4))
    for l in CONFIG["probe_layers"]:
        ax.plot(z[f"score_layer{l}"], label=f"layer {l}", alpha=0.8)
    ax.axhline(CALIBRATED_THRESHOLDS[CONFIG["probe_layers"][0]], color="red",
               linestyle=":", linewidth=0.8, label="example threshold")
    ax.set_xlabel("generated token index"); ax.set_ylabel("probe risk score")
    ax.set_title(f"Per-token risk trajectory — {example_files[0].stem}")
    ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(FIG_DIR / "example_trajectory.png", dpi=150); plt.close(fig)

# 18c: persistence sweep
if len(persistence_df) > 0:
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(persistence_df["k"], persistence_df["detection_rate"], marker="o", label="detection rate")
    ax.plot(persistence_df["k"], persistence_df["false_positive_rate"], marker="o", label="false positive rate")
    ax.set_xlabel("persistence window (consecutive tokens)"); ax.set_ylabel("rate")
    ax.set_title("Persistence sweep (Exp017 default thresholds)")
    ax.set_xticks(CONFIG["persistence_sweep"])
    ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(FIG_DIR / "persistence_sweep.png", dpi=150); plt.close(fig)

# 18d: latency
fig, ax = plt.subplots(figsize=(4, 4))
ax.bar(["raw", "firewall-monitored"],
       [latency_summary["raw_mean_s"], latency_summary["monitored_mean_s"]],
       yerr=[latency_summary["raw_stdev_s"], latency_summary["monitored_stdev_s"]])
ax.set_ylabel("seconds per 32-token generation")
ax.set_title(f"Latency (n={latency_summary['n_samples']})")
fig.tight_layout(); fig.savefig(FIG_DIR / "latency.png", dpi=150); plt.close(fig)

print(f"Saved figures under {FIG_DIR}: "
      + ", ".join(p.name for p in FIG_DIR.glob("*.png")))
mark_stage_done("stage18_figures")
save_manifest(RUN_DIR)

## Stage 19 — Final Forensic Report

In [ ]:
# Stage 19: auto-generated Markdown report combining every prior stage's
# numbers, plus every hash needed to reproduce this exact run. This is
# assembled entirely from values already computed above -- no new analysis.

def _safe(name, default="N/A"):
    return globals().get(name, default)

report_lines = []
report_lines.append(f"# NFW-001 Forensic Report — run_id `{CONFIG['run_id']}`")
report_lines.append(f"\nGenerated: {now_iso()}\n")

report_lines.append("## Reproducibility")
report_lines.append(f"- Dataset: `{CONFIG['hf_dataset_id']}` split=`{CONFIG['hf_split']}`, "
                     f"{NECENT_ROW_COUNT} rows at load time")
report_lines.append(f"- Split file SHA256: `{SPLIT_FILE_SHA256}`")
report_lines.append(f"- Model: `{CONFIG['model_id']}` "
                     f"(revision: `{RUN_MANIFEST['artifact_hashes'].get('model_revision')}`), "
                     f"hidden_size={QWEN_HIDDEN_SIZE}")
report_lines.append(f"- Probe layers: {CONFIG['probe_layers']}, pooling={CONFIG['pooling']}")
for l in CONFIG["probe_layers"]:
    report_lines.append(f"  - layer {l} artifact SHA256: "
                         f"`{RUN_MANIFEST['artifact_hashes'].get(f'probe_layer_{l}')}`")
report_lines.append(f"- Seed: {CONFIG['seed']}")
report_lines.append("- Package versions: " + json.dumps(RUN_MANIFEST["package_versions"]))

report_lines.append("\n## Split Composition")
report_lines.append(composition_df.to_markdown())

report_lines.append("\n## Calibration (calibration split ONLY)")
report_lines.append(f"- target_fpr: {CONFIG['target_fpr']}, vote_k: {CONFIG['vote_k']}")
report_lines.append(f"- Per-layer thresholds: {CALIBRATED_THRESHOLDS}")
report_lines.append(f"- Calibration-split ensemble FPR/TPR: "
                     f"{calib_ensemble_fpr:.4f} / {calib_ensemble_tpr:.4f}")

report_lines.append("\n## Frozen Evaluation — In-Distribution Test Splits")
report_lines.append(frozen_eval_df.to_markdown(index=False))

report_lines.append("\n## OOD Evaluation (frozen thresholds, not recalibrated)")
report_lines.append(ood_summary_df.to_markdown(index=False))

report_lines.append("\n## Behavioral Ground Truth Coverage")
report_lines.append(f"- behavioral_test: {n_total} rows, {n_labeled} labeled "
                     f"({n_missing} missing response_harmful — excluded, not imputed)")

report_lines.append("\n## Temporal Prediction Analysis (response-level)")
report_lines.append(f"- Evaluated on {n_eval} labeled+scored rows")
if n_eval > 0 and temporal_df["response_harmful_bool"].nunique() == 2:
    report_lines.append(f"- AUROC: {auroc:.4f}, AUPRC: {auprc:.4f}")
    report_lines.append(f"- Detection rate: {detection_rate:.4f}, FPR: {fpr:.4f}")
    report_lines.append(f"- Median lead time (tokens remaining after first exceedance): "
                         f"{lead_times.median() if len(lead_times) else float('nan'):.1f}")

report_lines.append("\n## Persistence Sweep")
report_lines.append(persistence_df.to_markdown(index=False) if len(persistence_df) else "N/A")

report_lines.append("\n## Firewall-Monitored Generation vs Behavioral Labels")
report_lines.append(f"- Evaluated on {n_fw_eval} rows")
if n_fw_eval > 0:
    report_lines.append(f"- Detection rate: {detection_rate:.4f}")
    report_lines.append(f"- False-block rate: {false_block_rate:.4f}")
report_lines.append("- A block is reported as detection only — NOT interpreted as harm "
                     "prevention, per NFW-001 rules.")

report_lines.append("\n## Latency")
report_lines.append(f"```json\n{json.dumps(latency_summary, indent=2)}\n```")

report_lines.append("\n## Notes / Warnings Logged During This Run")
for note in RUN_MANIFEST["notes"]:
    report_lines.append(f"- {note}")

report_md = "\n".join(str(x) for x in report_lines)
report_path = RUN_DIR / "FORENSIC_REPORT.md"
with open(report_path, "w") as f:
    f.write(report_md)

mark_stage_done("stage19_forensic_report")
save_manifest(RUN_DIR)
print(f"Forensic report written to {report_path} ({len(report_md)} chars)")

## Stage 20 — Save All Outputs to Drive

Everything in this notebook already writes directly to
`My Drive/NFW-001/runs/<run_id>/` as it's produced (that's what makes every
stage above resumable after a disconnect). This final cell just (a) writes the
last manifest update, (b) zips the run directory for easy download/sharing,
and (c) prints a manifest of exactly what's on disk.

In [ ]:
# Stage 20: final manifest write + zip + listing.
RUN_MANIFEST["completed_at"] = now_iso()
save_manifest(RUN_DIR)

zip_path = RUN_DIR.parent / f"{CONFIG['run_id']}.zip"
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in RUN_DIR.rglob("*"):
        if p.is_file():
            zf.write(p, p.relative_to(RUN_DIR.parent))
print(f"Zipped run directory to {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")

print(f"\nAll outputs under: {RUN_DIR}")
for p in sorted(RUN_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(RUN_DIR)}")

print(f"\nNFW-001 run '{CONFIG['run_id']}' complete. "
      f"Re-running this notebook in a fresh runtime with the same run_id will "
      f"resume from the last completed stage rather than redo everything.")